# 05 · Experimentos extras na GPU (nuvem)

1. **Checagem MPS × CUDA do artigo 1:** MedGemma 4B nas 40 imagens do piloto em CUDA (`run = piloto_cuda`), para
   comparar com as mesmas imagens já rodadas no Mac (`piloto_mps`). Só a checagem; nada da amostra de avaliação.
2. **MedGemma 27B:** checagem no piloto, depois os experimentos do artigo 2.

**Experimentos extras** (run = `a2`, separados dos arquivos do artigo 1):
- **Artigo 2, prompt em português completo:** `main_pt` na amostra de avaliação inteira das duas bases.
- **Artigo 2, 2×2 de língua** (subamostra de robustez, 300 imagens por base): instrução em português com formato de
  resposta em inglês (`pt_instr_en_fmt`) e instrução em inglês com formato em português (`en_instr_pt_fmt`). As outras
  duas células do 2×2 são `main_en`/`r1` (já rodado) e `main_pt`/`a2` (acima). O prefill `ICDR: ` é o mesmo nas quatro.

Rode pelo `rodar_nuvem.sh` com `NOTEBOOK=notebooks/05_extra_gpu.ipynb` (veja o README). Grava a cada imagem e retoma
de onde parou; no fim empacota `results/` e para a máquina. Tempo estimado na A100: ~2,5 h (≈1 s por imagem no 27B).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import cohort, infer, prompts
from vlmdr.runner import Runner

In [ ]:
import os, shutil, subprocess, tarfile, traceback
import torch, transformers
from huggingface_hub import constants as hf_constants, whoami

# etapas por modelo (o 4B só faz a checagem no piloto)
PLANO = {
    "medgemma_4b": ("piloto",),
    "medgemma_27b": ("piloto", "pt_completo", "dois_por_dois"),
}
MODELOS = os.environ.get("MODELOS", ",".join(PLANO)).split(",")
LIMITE = int(os.environ["LIMITE"]) if os.environ.get("LIMITE") else None   # ex.: 5 para um ensaio rápido
ENCERRAR = os.environ.get("ENCERRAR", "1") == "1"   # o rodar_nuvem.sh põe 0 e para a máquina ele mesmo
ENCERRAR_CMD = os.environ.get("ENCERRAR_CMD", "python3 ../gpubrasil.py parar")   # shutdown NÃO para a cobrança

assert torch.cuda.is_available(), "sem GPU CUDA: este notebook é só para a máquina da nuvem"
print("GPU:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9))
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| transformers", transformers.__version__)
print("disco livre (GB):", round(shutil.disk_usage(config.ROOT).free / 1e9), "| cache dos modelos:", hf_constants.HF_HUB_CACHE)
print("Hugging Face:", whoami()["name"], "| modelos:", MODELOS, "| limite:", LIMITE)
pr = prompts.load_prompts()
assert {"main_pt", "pt_instr_en_fmt", "en_instr_pt_fmt"} <= set(pr), "prompts do artigo 2 ausentes em settings/prompts.yaml"
cohorts = {ds: cohort.load(ds) for ds in ("brset", "mbrset")}
for ds, c in cohorts.items():
    falta = int((~c.image_path.map(lambda p: Path(p).exists())).sum())
    print(f"{ds}: {len(c)} imagens, {falta} não encontradas")
    assert falta == 0, f"imagens do {ds} não encontradas: confira {ds.upper()}_IMAGES"
runner = Runner(cohorts, "gpu_extra", limite=LIMITE, limpar_cache=len(MODELOS) > 1)

## Execução (empacota os resultados e para a máquina no fim, mesmo se der erro)

In [ ]:
try:
    for m in MODELOS:
        try:
            runner.roda(m, etapas=PLANO[m])
        except Exception as e:
            runner.registra(modelo=m, etapa="erro", problemas=f"{type(e).__name__}: {e}")
            traceback.print_exc()
finally:
    destino = config.ROOT / "resultados_nuvem.tar.gz"
    with tarfile.open(destino, "w:gz") as tar:
        tar.add(config.RESULTS, arcname="results")
    print("resultados em", destino, flush=True)
    if ENCERRAR:
        print("parando a máquina:", ENCERRAR_CMD, flush=True)
        subprocess.run(ENCERRAR_CMD, shell=True)
pd.DataFrame(runner.log)